## Worshop
## name: Sanjai S
## Reg no: 212223230186

In [8]:
# ================================
# 1. IMPORT LIBRARIES
# ================================
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import os

# ================================
# 2. DEVICE CONFIGURATION
# ================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ================================
# 3. DATA TRANSFORMS
# ================================
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# ================================
# 4. LOAD DATA
# ================================
data_dir = r"C:\Users\admin\Downloads\Cat-Dog_Pandas-20260609T142826Z-3-001\Cat-Dog_Pandas"

train_data = datasets.ImageFolder(
    r"C:\Users\admin\Downloads\Cat-Dog_Pandas-20260609T142826Z-3-001\Cat-Dog_Pandas",
    transform=train_transforms
)

test_data = datasets.ImageFolder(
   r"C:\Users\admin\Downloads\Cat-Dog_Pandas-20260609T142826Z-3-001\Cat-Dog_Pandas",
    transform=test_transforms
)
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)

class_names = train_data.classes
print("Classes:", class_names)

# ================================
# 5. LOAD PRETRAINED MODEL
# ================================
model = models.resnet18(pretrained=True)

# Freeze layers
for param in model.parameters():
    param.requires_grad = False

# Replace classifier
model.fc = nn.Sequential(
    nn.Linear(model.fc.in_features, 256),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(256, 3)  # 3 classes
)

model = model.to(device)

# ================================
# 6. LOSS & OPTIMIZER
# ================================
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

# ================================
# 7. TRAINING LOOP
# ================================
epochs = 10
best_acc = 0.0

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()

    train_acc = correct / len(train_data)

    print(f"Epoch [{epoch+1}/{epochs}] Loss: {running_loss:.4f} Accuracy: {train_acc:.4f}")

    # Save best model
    if train_acc > best_acc:
        best_acc = train_acc
        torch.save(model.state_dict(), "best_model.pth")

# ================================
# 8. EVALUATION
# ================================
model.eval()
correct = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)

        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()

test_acc = correct / len(test_data)
print("Test Accuracy:", test_acc)

# ================================
# 9. PREDICT SINGLE IMAGE
# ================================
from PIL import Image

def predict_image(image_path):
    model.eval()
    image = Image.open(image_path).convert("RGB")
    image = test_transforms(image).unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(image)
        _, pred = torch.max(output, 1)

    return class_names[pred.item()]

# Example
print(predict_image(r"C:\Users\admin\Downloads\Cat-Dog_Pandas-20260609T142826Z-3-001\Cat-Dog_Pandas\Train\panda\panda_00003.jpg"))

Using device: cuda
Classes: ['Test', 'Train', 'Valid']
Epoch [1/10] Loss: 79.6085 Accuracy: 0.6950
Epoch [2/10] Loss: 77.2605 Accuracy: 0.6997
Epoch [3/10] Loss: 76.7021 Accuracy: 0.7000
Epoch [4/10] Loss: 77.0266 Accuracy: 0.7000
Epoch [5/10] Loss: 76.0845 Accuracy: 0.7000
Epoch [6/10] Loss: 77.2759 Accuracy: 0.7000
Epoch [7/10] Loss: 75.7186 Accuracy: 0.7000
Epoch [8/10] Loss: 76.1248 Accuracy: 0.7000
Epoch [9/10] Loss: 75.3582 Accuracy: 0.7000
Epoch [10/10] Loss: 75.3138 Accuracy: 0.7000
Test Accuracy: 0.7
Train
